# Show an Overview of the 79NG Fjord Model Setup

Notebook by Markus Reinert (IOW, 2025/2026, https://orcid.org/0000-0002-3761-8029).

In [ ]:
from collections import namedtuple

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cmocean.cm as cmo
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from tools.configuration import Configuration
from tools.visualization import cm

## Load the model configuration

In [ ]:
config = Configuration()

## Load the topography

In [ ]:
filename = config.get_file_path("getm/domain/bathymetry")
assert filename == config.get_file_path("getm/ice/ice_file"), "filenames for bathymetry and ice thickness differ"
print(f"Loading topography {filename!r}.")
ds = xr.open_dataset(filename)

# Use this aspect ratio in lat–lon plots for squared grid cells; since dx and dy
# are approximately equal, this gives an approximate equal-area map.
grille_carree = ds.dlon / ds.dlat

# Compute the ice mask
ice_mask = ds.glIceD > 0

ds

## Load the grounding line

In [ ]:
filename = config.get_file_path("getm/rivers/river_info")
n_gline = 0
discharges = {}
print(f"Loading grounding line information {filename!r}.")
with open(filename) as f:
    for line in f.readlines():
        # If the line contains a comment, remove it.
        if "#" in line:
            line = line[:line.index("#")]
        line = line.strip()
        if not line:
            continue
        if not n_gline:
            n_gline = int(line)
            continue
        i, j, name = line.split()
        if name not in discharges:
            discharges[name] = ([], [])
        # Subtract one from the indices because GETM starts counting at 1 and not 0
        discharges[name][0].append(int(i) - 1)
        discharges[name][1].append(int(j) - 1)
assert n_gline == sum(len(discharge[0]) for discharge in discharges.values()), "number of discharge points is inconsistent"
if len(discharges) == 0:
    print("There is no grounding line.")
elif len(discharges) == 1:
    print(f"There is 1 grounding line consisting of {n_gline} grid points.")
else:
    print(f"There are {len(discharges)} grounding lines consisting of {n_gline} grid points in total.")

### Make step-wise grounding line

To show the grounding line(s) in a step-wise manner, we determine the lat–lon coordinates of the western boundary of the grid cells that belong to the grounding line(s).

In [ ]:
glines_lat = {}
glines_lon = {}
for name, indices in discharges.items():
    glines_lat[name] = []
    glines_lon[name] = []
    for i, j in zip(*indices):
        glines_lat[name] += [ds.lat[j] - ds.dlat / 2, ds.lat[j] + ds.dlat / 2]
        glines_lon[name] += [ds.lon[i] - ds.dlon / 2, ds.lon[i] - ds.dlon / 2]

## Load the boundary conditions

In [ ]:
filename = config.get_file_path("getm/m3d/bdyfile_3d")
print(f"Loading boundary conditions {filename!r}.")
bdy_3D = xr.open_dataset(filename)
# Remove time axis if there are only 2 points in time with equal data
assert bdy_3D.time.size == 2, "boundary file does not have exactly 2 points in time"
assert bdy_3D.isel(time=0, drop=True).equals(bdy_3D.isel(time=1, drop=True)), "boundary data differ between the 2 points in time"
bdy_3D = bdy_3D.isel(time=0, drop=True)
bdy_3D

### Get start and end indices of the open boundaries

In [ ]:
# North
# From the first ocean point (mask is True) to one before the last grid point,
# because the last grid point (NE corner) belongs to the eastern boundary
i_bdy_start_N = np.where(ds.mask.isel(lat=-1))[0][0]
i_bdy_end_N = ds.lon.size - 2

# East
# Full extent of the model grid
i_bdy_start_E = 0
i_bdy_end_E = ds.lat.size - 1

# South
# Analog to the northern boundary
i_bdy_start_S = np.where(ds.mask.isel(lat=0))[0][0]
i_bdy_end_S = ds.lon.size - 2

### Get number of boundary points

In [ ]:
n_bdy_N = i_bdy_end_N - i_bdy_start_N + 1
n_bdy_E = i_bdy_end_E - i_bdy_start_E + 1
n_bdy_S = i_bdy_end_S - i_bdy_start_S + 1
n_bdy_points = n_bdy_N + n_bdy_E + n_bdy_S
print("Open boundary consists of", n_bdy_points, "grid points.")

### Extract boundary coordinates and bathymetry

In [ ]:
z_seabed = -np.concatenate([
    ds.bathymetry.isel(lat=0, lon=slice(i_bdy_start_S, i_bdy_end_S + 1)),
    ds.bathymetry.isel(lon=-1),
    ds.bathymetry.isel(lat=-1, lon=slice(i_bdy_end_N, i_bdy_start_N - 1, -1)),
])
l_bdy = np.concatenate([
    ds.lon[i_bdy_start_S : i_bdy_end_S + 1],
    ds.lat,
    ds.lon[i_bdy_end_N : i_bdy_start_N - 1 : -1],
])

### Get the size of the sponge zone

The sponge zone consists of the open boundary points and the `n_sponge` adjacent points.
In the sponge zone, the bathymetry is equal to that of the first grid point in the model domain outside the sponge zone.

In [ ]:
n_sponge = int(config.get_text("getm/m3d/bdy3d_sponge_size"))
print(f"Sponge zone is {n_sponge} (+ 1) grid points wide.")

## Prepare the figure

In [ ]:
lon_min = ds.lon[0] - ds.dlon/2
lon_max = ds.lon[-1] + ds.dlon/2
lat_min = ds.lat[0] - ds.dlat/2
lat_max = ds.lat[-1] + ds.dlat/2

In [ ]:
plt.rcParams["axes.titlesize"] = "medium"
plt.rcParams["axes.labelsize"] = "small"
plt.rcParams["xtick.labelsize"] = "small"
plt.rcParams["ytick.labelsize"] = "small"

In [ ]:
Variable = namedtuple("Variable", ["letter", "data", "cmap", "label", "ticks"])
variables = [
    Variable("d", bdy_3D.salt, cmo.haline, "salinity in g kg$^{-1}$", np.arange(31, 35)),
    Variable("e", bdy_3D.temp, cmo.thermal, "temperature in °C", np.arange(-1, 3)),
]

In [ ]:
ocean = "#003d8c"
feature_ocean = cfeature.NaturalEarthFeature("physical", "ocean", "10m", edgecolor="none", facecolor=ocean)

### Get aspect ratios of the maps

In [ ]:
latlon = ccrs.PlateCarree()

In [ ]:
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.LambertAzimuthalEqualArea(-45, 75)})
ax.set_extent([-57, -29, 59.5, 84], latlon)
fig.canvas.draw()
plt.close(fig)
ratio_map = ax.get_window_extent().width / ax.get_window_extent().height
print("Aspect ratio:", ratio_map)

In [ ]:
fig, ax = plt.subplots()
im = ds.bathymetry.plot(ax=ax, cmap=cmo.deep, vmin=0, add_colorbar=False)
ax.set(xlim=(-22.8, None), aspect=grille_carree)
fig.canvas.draw()
plt.close(fig)
ratio_model = ax.get_window_extent().width / ax.get_window_extent().height
print("Aspect ratio:", ratio_model)

In [ ]:
fig, ax = plt.subplots()
ds.mask.plot.contour(ax=ax, levels=[True / 2], colors=["0.5"], linewidths=1)
im = ds.glIceD.where(ice_mask).plot(ax=ax, vmin=0, cmap=cmo.ice_r, add_colorbar=False)
ax.set(xlim=(None, -19.2), ylim=(79.25, 79.85), aspect=grille_carree)
fig.canvas.draw()
plt.close(fig)
ratio_ice = ax.get_window_extent().width / ax.get_window_extent().height
print("Aspect ratio:", ratio_ice)

## Create the figure

In [ ]:
fig, axs = plt.subplot_mosaic(
    "abc\nddd\neee",
    height_ratios=[2.3, 1, 1], width_ratios=[ratio_map, ratio_model, ratio_ice],
    per_subplot_kw={"a": dict(projection=ccrs.LambertAzimuthalEqualArea(-45, 75))},
    figsize=(18*cm, 15*cm), dpi=150,
)


ax = axs["a"]
ax.set_extent([-57, -29, 59.5, 84], latlon)
ax.add_feature(feature_ocean)
cl = ax.coastlines(resolution="10m", linewidth=0.5)
zorder = cl.get_zorder()
gl = ax.gridlines(draw_labels=["left", "top", "bottom"], linestyle=":", linewidth=1, zorder=zorder + 2)
gl.xlocator = mticker.FixedLocator([-80, -40, 0])
gl.ylabel_style = {"size": "small"}
gl.xlabel_style = {"size": "small"}
# Mark the model domain
col = "tab:red"
ax.plot(
    [lon_min, lon_max, lon_max, lon_min, lon_min],
    [lat_min, lat_min, lat_max, lat_max, lat_min],
    col, linewidth=1.5, transform=latlon, zorder=zorder + 3,
)
ds.mask.plot.contourf(ax=ax, levels=[0.5, 1], colors=["w", ocean], add_colorbar=False, zorder=zorder + 1, transform=latlon)
kwargs = dict(ha="right", va="top", color=col, zorder=zorder + 3, transform=latlon)
arrowprops = {"arrowstyle": "->", "color": col, "shrinkA": 1, "shrinkB": 1}
ax.annotate("79NG\nfjord", (lon_min, lat_min), (-30, 76.5), **kwargs, arrowprops=arrowprops)


ax = axs["b"]
ax.set_facecolor("0.8")

# Show the bathymetry
im = ds.bathymetry.plot(ax=ax, cmap=cmo.deep, vmin=0, add_colorbar=False)
cax = ax.inset_axes((0.03, 0.38, 0.025, 0.48))
fig.colorbar(im, cax, ticks=np.arange(0, 1000, 250), label="depth in m")
cax.invert_yaxis()

# Mark the calving front
col = "darkblue"
ds.glIceD.where(ds.mask).plot.contour(ax=ax, levels=[0], colors=col, linewidths=1)
kwargs = dict(color=col, size="small", ha="center")
arrowprops = {"arrowstyle": "->", "color": col, "relpos": (0.25, 0), "shrinkA": 0, "patchA": None}
ax.annotate("calving\nfront", (-19.35, 79.68), (-18.8, 79.84), arrowprops=arrowprops, **kwargs)
ax.annotate("calving\nfront", (-20.05, 79.76), (-18.8, 79.84), arrowprops=arrowprops, **kwargs)

# Mark the grounding line
col = "tab:red"
for discharge in discharges:
    ax.plot(glines_lon[discharge], glines_lat[discharge], col, lw=1, solid_capstyle="butt")
ax.annotate(
    "grounding line",
    (glines_lon[discharge][0], glines_lat[discharge][0]),
    (-22.1, 79.2),
    arrowprops={"arrowstyle": "->", "color": col, "relpos": (-0.01, 0.5), "shrinkA": 0, "shrinkB": 0, "patchA": None},
    color=col, size="small",
)
ax.set_xlim(-22.8, None)

# Mark the open boundary and the n_sponge adjacent grid cells
col = "k"
kwargs = dict(hatch="\\"*8, facecolor="none", lw=0, edgecolor=col)
# southern boundary
ax.fill_between(ds.lon - ds.dlon/2, ds.lat[0] - ds.dlat/2, ds.lat[n_sponge] + ds.dlat/2, ds.mask[0], **kwargs)
# northern boundary
ax.fill_between(ds.lon - ds.dlon/2, ds.lat[-1] + ds.dlat/2, ds.lat[-1 - n_sponge] - ds.dlat/2, ds.mask[-1], **kwargs)
# eastern boundary
ax.fill_between(ds.lon[-2 - n_sponge:] + ds.dlon/2, ds.lat[0] - ds.dlat/2, ds.lat[-1] + ds.dlat/2, **kwargs)
ax.annotate(
    "open boundary",
    (ds.lon[np.where(ds.mask[-1])[0][0]], ds.lat[-n_sponge]),
    (-19.9, 80.26),
    arrowprops={"arrowstyle": "->", "color": col, "relpos": (1, 0.5), "shrinkA": 0, "shrinkB": 0},
    color=col, size="small",
)
ax.xaxis.set_major_formatter(lambda x, pos: f"{-x:.0f}°W")
ax.yaxis.set_major_formatter(lambda x, pos: f"{x}°N")
ax.set(title="Model domain and bathymetry", xlabel="", ylabel="", aspect=grille_carree)


ax = axs["c"]
ax.set_facecolor("0.8")
ds.mask.where(ds.mask).plot(ax=ax, vmax=2, cmap="Greys", add_colorbar=False)
im = ds.glIceD.where(ice_mask).plot(ax=ax, vmin=0, cmap=cmo.ice_r, add_colorbar=False)
cbar = fig.colorbar(im, ax.inset_axes((0.04, 0.86, 0.50, 0.03)), orientation="horizontal", ticks=np.arange(0, 750, 250))
cbar.set_label("ice thickness\nin m", labelpad=2)
ax.xaxis.set_major_formatter(lambda x, pos: f"{-x:.0f}°W")
ax.yaxis.set_major_formatter(lambda x, pos: f"{round(x, 5)}°N")
ax.set(title="Glacier tongue", xlabel="", ylabel="", xlim=(None, -19.2), ylim=(79.25, 79.85), aspect=grille_carree)


for var in variables:
    ax = axs[var.letter]
    # Re-assemble the boundary data in counter-clockwise direction from south to north
    data = np.concatenate((var.data[n_bdy_N + n_bdy_E:], var.data[n_bdy_N : n_bdy_N + n_bdy_E], var.data[n_bdy_N - 1 : : -1])).T
    print(f"Range of {var.label}: {data.min():.3f} to {data.max():.3f}, ticks: {var.ticks}")
    im = ax.pcolormesh(bdy_3D.nbdy, bdy_3D.zax, data, cmap=var.cmap)
    fig.colorbar(im, orientation="horizontal", label=var.label, ticks=var.ticks, cax=ax.inset_axes((0.4, 0.45, 0.25, 0.06)))
    # Hide everything below the seabed
    ax.fill_between(bdy_3D.nbdy, z_seabed, z_seabed.min(), color="0.8")
    ax.plot(bdy_3D.nbdy, z_seabed, "k", lw=1)
    # Mark the transitions between sections
    kwargs = dict(color="r", lw=1.5, ls="--")
    ax.axvline(n_bdy_S - 0.5, **kwargs)
    ax.axvline(n_bdy_S + n_bdy_E - 0.5, **kwargs)
    ax.set_ylim(z_seabed.min(), 0)
    ax.set_ylabel("depth in m")
    ax.yaxis.set_major_formatter(lambda x, pos: -int(x))
    # Put x-labels at each degree longitude and at every fifth degree latitude
    indices = (
        [np.argmin(abs(l_bdy[:n_bdy_S] + lon)) for lon in range(19, 14, -1)] +
        [np.argmin(abs(l_bdy - lat)) for lat in np.arange(79.2, 80.3, 0.2)] + 
        [np.argmin(abs(l_bdy[n_bdy_S + n_bdy_E:] + lon)) + n_bdy_S + n_bdy_E for lon in (15, 16)]
    )
    labels = [f"{-l:.0f}°W" if l < 0 else f"\n{l:.1f}°N" for l in l_bdy[indices]]
    ax.set_xticks(indices, labels if var.letter == "e" else [])
    if var.letter == "d":
        ax.set_title("Boundary conditions")
    else:
        ax.set_xlabel("open boundary from south over east to north (counter-clockwise)")


text = {
    "a": (0.0400, 0.978, "(a)"),
    "b": (0.0200, 0.978, "(b)"),
    "c": (0.0250, 0.978, "(c)"),
    "d": (0.0065, 0.965, "(d)"),
    "e": (0.0065, 0.965, "(e)"),
}
kwargs = dict(
    va="top", size="small", clip_on=True, zorder=zorder + 3,
    bbox={"boxstyle": "square", "facecolor": "w", "alpha": 2/3, "edgecolor": "none"},
)
for letter, ax in axs.items():
    ax.text(*text[letter], transform=ax.transAxes, **kwargs)

fig.suptitle("Overview of the 79° North Glacier (79NG) fjord model setup", weight="bold")
fig.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.09, wspace=0.32, hspace=0.06)
fig.savefig("figures/fig_1_overview_setup.png", dpi=600)